# Realtime Agents

A normal `Agent` works in turns: you send a request, wait, and get a full answer. A **`RealtimeAgent`** works like a
phone call: one connection (a WebSocket) stays open, and audio and events flow both ways, so the agent can answer fast
and even be interrupted. Here we build a "Mission Dispatch Hotline".

`RealtimeRunner` opens the session, and you read what happens through a stream of **events** (a tool starting, a tool
ending, the agent finishing). To keep it simple, we run text-only, so no microphone is needed.

The SDK's default realtime model is `gpt-realtime-2.1`. For offline tests, `RealtimeRunner(agent, model=...)` also
accepts your own `RealtimeModel` class, just like `ScriptedModel` does for normal agents. This episode runs a real
session, so it needs a real key.


## Give the hotline a tool

Realtime agents use the same `@function_tool` as normal agents.


In [2]:
from agents import (
    function_tool,  # turns a normal function into a tool an Agent can call
)


# Define the tool
# @function_tool callback: gives the live session a real tool call to make
@function_tool
def get_ninja_rank(name: str) -> str:
    """Look up a ninja's rank in the Hidden Leaf Village registry."""
    return f"{name}: Kage-level"

## Run a live text-only session — needs a real key

`RealtimeRunner(agent).run(...)` gives a session. We send one message, then watch the events: `tool_start` and
`tool_end` show the tool call, and `agent_end` means the agent finished a turn.

**Gotcha, confirmed:** session settings go under `model_config={"initial_model_settings": {...}}`. A wrong key like
`{"session": {...}}` is quietly ignored. `asyncio.timeout` stops the waiting, so the cell never hangs. Skipped when there
is no `OPENAI_API_KEY`.


In [ ]:
import asyncio  # limits how long the live session may wait for events
import os  # reads OPENAI_API_KEY from the environment

from agents.realtime import (
    RealtimeAgent,  # like Agent, but for a live, always-open conversation
    RealtimeRunner,  # runs a RealtimeAgent's live conversation
)
from dotenv import load_dotenv  # loads the .env file into the environment

load_dotenv()


# Demo: a real text-only realtime session that calls a tool and then answers
async def demo_live() -> None:
    if not os.environ.get("OPENAI_API_KEY"):
        print("[skipped] add a real OPENAI_API_KEY to .env to run this one for real")
        return

    # Initialize the agent
    agent = RealtimeAgent(
        name="Mission Dispatch Hotline",
        instructions="Always call get_ninja_rank before answering a question about a ninja. Answer briefly.",
        tools=[get_ninja_rank],
    )
    # Run the Agent
    runner = RealtimeRunner(agent)
    # modalities=["text"] removes audio, so no microphone or speakers are needed
    session = await runner.run(
        model_config={
            "initial_model_settings": {
                "modalities": ["text"],  # removes audio
            }
        }
    )

    history: list = []
    tool_finished = False
    async with session:
        await session.send_message("What rank is Naruto Uzumaki?")
        try:
            async with asyncio.timeout(60):
                async for event in session:
                    # raw_model_event is low-level network traffic, skipped to keep the printout easy to read
                    if event.type == "raw_model_event":
                        continue
                    if event.type in ("tool_start", "tool_end", "agent_end"):
                        print("[event]", event.type)
                    if event.type == "tool_end":
                        # proves the live session really called our tool
                        print("[tool result]", event.output)
                        tool_finished = True
                    if event.type == "history_updated":
                        history = event.history
                    # the first agent_end closes the tool-call turn, the one after tool_end closes the answer
                    if event.type == "agent_end" and tool_finished:
                        break
        except TimeoutError:
            print("[stopped waiting after 60s]")

    # Inspect the result
    for item in history:
        # item.type is "message", because we dont want tool call result
        if item.type == "message" and item.role == "assistant":
            # item.content will be a list
            for part in item.content:
                # message (assistant)
                #     ├── part: audio   ← voice data
                #     └── part: text    ← uska transcript
                if part.type == "text" and part.text:
                    print("[assistant reply]", part.text)
                    print("-" * 50)


await demo_live()
# Outside a notebook: asyncio.run(demo_live())

--------------------------------------------------
--------------------------------------------------
--------------------------------------------------
--------------------------------------------------
--------------------------------------------------
--------------------------------------------------
--------------------------------------------------
[event] agent_end
--------------------------------------------------
[event] tool_start
--------------------------------------------------
--------------------------------------------------
[event] tool_end
[tool result] Naruto Uzumaki: Kage-level
--------------------------------------------------
--------------------------------------------------
--------------------------------------------------
--------------------------------------------------
--------------------------------------------------
[event] agent_end
[assistant reply] I’ll quickly check the registry for his official rank.
-------------------------------------------------

## Summary

- `RealtimeAgent` and `RealtimeRunner` run a session that stays open, and you follow it through events like
  `tool_start`, `tool_end`, and `agent_end`.
- `initial_model_settings={"modalities": ["text"]}` makes a text-only session, so you can try realtime with no
  microphone. For offline tests you can pass your own `RealtimeModel`.
